In [ ]:
!pip install mediapipe

In [ ]:
import tensorflow as tf

build_info = tf.sysconfig.get_build_info()
print("Build Information: ", build_info)

In [ ]:
print("GPU Available: ", tf.test.is_gpu_available())
print("GPU Name: ", tf.config.experimental.list_physical_devices('GPU'))

## Expression Recognition

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from tensorflow.keras.models import load_model

mp_hands = mp.solutions.hands
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

expression_model = load_model("expression_model.h5")
def preprocess_face(frame, face_landmarks):
    h, w, _ = frame.shape
    x_coords = [int(lm.x * w) for lm in face_landmarks.landmark]
    y_coords = [int(lm.y * h) for lm in face_landmarks.landmark]
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    face_image = frame[y_min:y_max, x_min:x_max]
    face_image = cv2.resize(face_image, (48, 48))
    face_image = cv2.cvtColor(face_image, cv2.COLOR_BGR2GRAY)
    face_image = face_image / 255.0
    face_image = np.expand_dims(face_image, axis=-1)
    face_image = np.expand_dims(face_image, axis=0)
    return face_image

def count_fingers(hand_landmarks):
    tip_ids = [4, 8, 12, 16, 20] 
    finger_count = 0

    if hand_landmarks.landmark[tip_ids[0]].x < hand_landmarks.landmark[tip_ids[0] - 1].x:
        finger_count += 1

    for id in range(1, 5):
        if hand_landmarks.landmark[tip_ids[id]].y < hand_landmarks.landmark[tip_ids[id] - 2].y:
            finger_count += 1

    return finger_count

cap = cv2.VideoCapture(0)

with mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7) as hands, \
     mp_face_mesh.FaceMesh(min_detection_confidence=0.7, min_tracking_confidence=0.7) as face_mesh:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False

        hand_results = hands.process(frame_rgb)
        face_results = face_mesh.process(frame_rgb)

        if hand_results.multi_hand_landmarks:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style(),
                )
                finger_count = count_fingers(hand_landmarks)
                cv2.putText(frame, f"Fingers: {finger_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        if face_results.multi_face_landmarks:
            for face_landmarks in face_results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    face_landmarks,
                    mp_face_mesh.FACEMESH_CONTOURS,
                    mp_drawing_styles.get_default_face_mesh_tesselation_style(),
                    mp_drawing_styles.get_default_face_mesh_contours_style(),
                    mp_drawing_styles.get_default_face_mesh_iris_connections_style(),
                )

                face_image = preprocess_face(frame, face_landmarks)
                expression_prediction = expression_model.predict(face_image)
                expression_label = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"][np.argmax(expression_prediction)]

                cv2.putText(frame, f"Expression: {expression_label}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow('Hand and Face Detection', frame)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

## Hand Gesture Movement

In [ ]:
mp_hands = mp.solutions.hands
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

cap = cv2.VideoCapture(0)

with mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7) as hands, \
     mp_face_mesh.FaceMesh(min_detection_confidence=0.7, min_tracking_confidence=0.7) as face_mesh:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False

        hand_results = hands.process(frame_rgb)
        face_results = face_mesh.process(frame_rgb)

        if hand_results.multi_hand_landmarks:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style(),
                )

        if face_results.multi_face_landmarks:
            for face_landmarks in face_results.multi_face_landmarks:

                mp_drawing.draw_landmarks(
                    frame,
                    face_landmarks,
                    None,  
                    mp_drawing_styles.get_default_face_mesh_tesselation_style(),
                    mp_drawing_styles.get_default_face_mesh_contours_style(),
                    mp_drawing_styles.get_default_face_mesh_iris_connections_style(),
                )

        cv2.imshow('Hand and Face Detection', frame)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()